<a href="https://colab.research.google.com/github/jane-rusakova/python_for_hw_tasks/blob/main/HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# параметри
p1 = 0.19          # базове утримання
p2 = 0.20          # утримання після змін (MDE = +1%)
alpha = 0.05
power = 0.80

# ефект (Cohen's h для пропорцій)
effect_size = proportion_effectsize(p1, p2)

# розрахунок розміру вибірки
analysis = NormalIndPower()
sample_size_per_group = analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1
)

total_sample_size = sample_size_per_group * 2

sample_size_per_group, total_sample_size


(24637.863136236123, 49275.726272472246)

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [2]:
import pandas as pd

# зчитуємо дані
df = pd.read_csv("cookie_cats.csv")

# середнє утримання на 7 день по версіях гри
retention_7_by_version = (
    df
    .groupby("version")["retention_7"]
    .mean()
)

retention_7_by_version


,retention_7
version,
gate_30,0.190201
gate_40,0.182000


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [3]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# 1. Розділяємо дані на контроль і тест
control = df[df["version"] == "gate_30"]["retention_7"]
treatment = df[df["version"] == "gate_40"]["retention_7"]

# 2. Кількість успіхів і розміри вибірок
count = [
    control.sum(),
    treatment.sum()
]
nobs = [
    control.count(),
    treatment.count()
]

# 3. Z-test для двох пропорцій (двосторонній)
z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="two-sided"
)

# 4. Довірчі інтервали 95%
ci_control = proportion_confint(
    count=control.sum(),
    nobs=control.count(),
    alpha=0.05,
    method="normal"
)

ci_treatment = proportion_confint(
    count=treatment.sum(),
    nobs=treatment.count(),
    alpha=0.05,
    method="normal"
)

# 5. Вивід результатів у потрібному форматі
print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")


z statistic: 3.1644
p-value: 0.001554
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


### Висновок

Для порівняння утримання користувачів на 7-й день між версіями гри
було використано z-test для двох пропорцій на рівні значущості α = 0.05.


Оскільки p-value > 0.05, ми не відхиляємо нульову гіпотезу H₀.
Це означає, що статистично значущої різниці між retention_7
у версіях gate_30 та gate_40 не виявлено.


95% довірчі інтервали для показника retention_7 у контрольній
та тестовій групах перетинаються.

Перетин довірчих інтервалів означає, що можливі значення
утримання користувачів для двох версій гри суттєво перекриваються,
що узгоджується з результатом статистичного тесту та вказує
на відсутність переконливого ефекту від переміщення воріт.


Переміщення воріт з 30 на 40 рівень не призвело до статистично
значущих змін у показнику утримання користувачів на 7-й день
після встановлення гри.


4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


### Формулювання гіпотез для χ²-тесту

Нульова гіпотеза (H₀):
Між версією гри (gate_30 / gate_40) та утриманням гравця
на 7-й день після реєстрації немає залежності.
Змінні є незалежними.

Альтернативна гіпотеза (Hₐ):
Між версією гри та утриманням гравця на 7-й день
існує статистично значуща залежність.


In [4]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. Контингентна таблиця
contingency_table = pd.crosstab(df["version"], df["retention_7"])

contingency_table


retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


In [5]:
# 2. χ²-тест
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

chi2_stat, p_value


(np.float64(9.959086799559167), np.float64(0.0016005742679058301))

In [6]:
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")


Chi-square statistic: 9.9591
p-value: 0.001601
Degrees of freedom: 1


### Висновок

Для перевірки залежності між версією гри та утриманням гравця
на 7-й день було використано χ²-тест на рівні значущості α = 0.05.

Оскільки p-value > 0.05, ми не відхиляємо нульову гіпотезу H₀.

Це означає, що статистично значущої залежності між версією гри
та утриманням гравця на 7-й день після реєстрації не виявлено.
